# I. Sequence-based features

In [18]:
from Bio import SeqIO
from collections import Counter
from propy import PyPro

def calculate_aac(sequence: str) -> dict:
    """Calculate AAC with all 20 standard amino acids"""
    standard_amino_acids = list('ACDEFGHIKLMNPQRSTVWY')
    aac = Counter(sequence)
    total_length = len(sequence)

    # Создаем словарь со всеми аминокислотами, включая отсутствующие
    result = {}
    for aa in standard_amino_acids:
        result[aa] = aac.get(aa, 0) / total_length

    return result

def calculate_dpc(sequence: str) -> dict:
    """Calculate DPC with all possible dipeptides"""
    standard_amino_acids = list('ACDEFGHIKLMNPQRSTVWY')

    # Создаем все возможные дипептиды
    all_dipeptides = [a + b for a in standard_amino_acids for b in standard_amino_acids]

    dpc = {}
    total_pairs = len(sequence) - 1

    # Считаем частоты для существующих дипептидов
    for i in range(len(sequence) - 1):
        dipeptide = sequence[i:i+2]
        dpc[dipeptide] = dpc.get(dipeptide, 0) + 1

    # Создаем полный словарь со всеми дипептидами
    result = {}
    for dp in all_dipeptides:
        result[dp] = dpc.get(dp, 0) / total_pairs if total_pairs > 0 else 0

    return result

def calculate_qso(sequence, max_lag=30, weight=0.1):
    """Calculate Quasi-sequence-order descriptors"""
    return PyPro.GetProDes(sequence).GetQSO(maxlag=max_lag, weight=weight)

# II. Physicochemical features

In [20]:
import numpy as np

physicochemical_groups: dict[str, dict[str, int]] = {
    'hydrophobicity': {
        'A': 1, 'R': -1, 'N': -1, 'D': -1, 'C': 1, 'Q': -1,
        'E': -1, 'G': 0, 'H': -1, 'I': 1, 'L': 1, 'K': -1,
        'M': 1, 'F': 1, 'P': 0, 'S': -1, 'T': -1, 'W': 1,
        'Y': -1, 'V': 1
    },
    'polarity': {
        'A': 0, 'R': 1, 'N': 1, 'D': 1, 'C': 0, 'Q': 1,
        'E': 1, 'G': 0, 'H': 1, 'I': 0, 'L': 0, 'K': 1,
        'M': 0, 'F': 0, 'P': 0, 'S': 1, 'T': 1, 'W': 0,
        'Y': 1, 'V': 0
    },
}


def calculate_physicochemical(sequence, properties):
    """Calculate physicochemical properties composition"""
    results = {}
    for prop_name, prop_dict in properties.items():
        prop_values = [prop_dict.get(aa, 0) for aa in sequence]
        results[prop_name] = {
            'mean': np.mean(prop_values),
            'std': np.std(prop_values),
            'composition': Counter(prop_values)
        }
    return results


def calculate_ctdt(sequence, property_dict):
    """Calculate Composition/Transition/Distribution descriptors"""
    # Convert sequence to property values
    prop_values = [property_dict.get(aa, 0) for aa in sequence]

    # Composition: percentage of each class
    composition = Counter(prop_values)
    total = len(sequence)
    comp_result = {
        f"comp_{cls}": count/total for cls, count in composition.items()
    }

    # Transition: transitions between classes
    transition_result = {}
    for i in range(len(prop_values) - 1):
        cls_pair = tuple(sorted([prop_values[i], prop_values[i+1]]))
        if cls_pair[0] != cls_pair[1]:  # Only count transitions between different classes
            transition_result[cls_pair] = transition_result.get(cls_pair, 0) + 1

    # Normalize transitions
    total_transitions = len(sequence) - 1
    trans_result = {f"trans_{cls1}_{cls2}": count/total_transitions
                   for (cls1, cls2), count in transition_result.items()}

    # Distribution: distribution of property values along sequence
    # This would typically be calculated as 5-point distribution (0%, 25%, 50%, 75%, 100%)
    return {**comp_result, **trans_result}

In [ ]:
import csv

def flatten_dict(d, parent_key='', sep='_'):
    """
    Преобразует вложенный словарь в плоский словарь с объединенными ключами
    """
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        elif isinstance(v, list):
            # Для списков создаем отдельные колонки с индексами
            for i, item in enumerate(v):
                items.append((f"{new_key}{sep}{i}", item))
        else:
            items.append((new_key, v))
    return dict(items)

def features_to_csv(features_list, output_filename):
    """
    Записывает все извлеченные признаки в CSV файл

    Args:
        features_list: список словарей с признаками для каждой последовательности
        output_filename: имя выходного CSV файла
    """
    if not features_list:
        return

    # Преобразуем все вложенные словари в плоские
    flattened_features = []
    for feature_dict in features_list:
        flattened = flatten_dict(feature_dict)
        flattened_features.append(flattened)

    # Получаем все уникальные ключи (названия столбцов)
    all_keys = set()
    for flattened in flattened_features:
        all_keys.update(flattened.keys())

    # Сортируем ключи для единообразия
    sorted_keys = sorted(all_keys)

    # Записываем в CSV
    with open(output_filename, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=sorted_keys)
        writer.writeheader()

        for flattened in flattened_features:
            # Заполняем отсутствующие значения пустыми строками
            row = {key: flattened.get(key, '') for key in sorted_keys}
            writer.writerow(row)

    print(f"Признаки успешно записаны в {output_filename}")

# Извлечение признаков из датасета в файл csv

In [22]:
path_train_neg = "Datasets/T6SE_Training_Neg_1112.fasta"
path_train_pos = "Datasets/T6SE_Training_Pos_138.fasta"

In [ ]:
def get_label(id: str) -> int:
    if "Non-effector" in id:
        return 0
    else:
        return 1

In [93]:
def record_features(path, label=None) -> None:
    all_features = []
    for record in SeqIO.parse(path, "fasta"):
        seq_str = str(record.seq)
        features = {
            'id': record.id,
            'aac': calculate_aac(seq_str),
            'dpc': calculate_dpc(seq_str),
            'qso': calculate_qso(seq_str),
            'physicochemical': calculate_physicochemical(seq_str, physicochemical_groups),
            'ctdt': {
                prop: calculate_ctdt(seq_str, prop_dict) for prop, prop_dict in physicochemical_groups.items()
            },
            'label': get_label(record.id) if label is None else label
        }
        all_features.append(features)
    features_to_csv(all_features, f"{path}.csv")

In [94]:
record_features('Datasets/Test.fasta')
record_features(path_train_pos, 1)
record_features(path_train_neg, 0)

Признаки успешно записаны в Datasets/Test.fasta.csv
Признаки успешно записаны в Datasets/T6SE_Training_Pos_138.fasta.csv
Признаки успешно записаны в Datasets/T6SE_Training_Neg_1112.fasta.csv


In [95]:
import pandas as pd

def read_data_csv(path):
    return pd.read_csv(
		path,
		sep=',',
		encoding='utf-8',
		header=0
	)

def check_data(dataframe):
	print("Есть ли пустые значения в данных:", dataframe.isnull().values.any())

	print("Количество пустых значений по столбцам:")
	print(dataframe.isnull().sum())

	print("Доля пустых значений по столбцам:")
	print(dataframe.isnull().mean())

	print("Информация о DataFrame:")
	dataframe.info()

In [96]:
positive_df = read_data_csv('Datasets/T6SE_Training_Pos_138.fasta.csv')
negative_df = read_data_csv('Datasets/T6SE_Training_Neg_1112.fasta.csv')
test_df = read_data_csv('Datasets/Test.fasta.csv')

In [97]:
print(len(negative_df[negative_df['label'] == 1]))

0


# Построение классификатора

In [98]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt

In [110]:
# Объединение датасетов
df = pd.concat([positive_df, negative_df], axis=0, ignore_index=True)
df = df.drop('id', axis=1)

# Разделение на признаки и целевую переменную
X = df.drop('label', axis=1)
Y = df['label']

# Проверка размерности
print(f"Размерность данных: {X.shape}")
print(f"Количество положительных примеров: {sum(Y == 1)}")
print(f"Количество отрицательных примеров: {sum(Y == 0)}")

Размерность данных: (1250, 538)
Количество положительных примеров: 138
Количество отрицательных примеров: 1112


In [111]:
# Для борьбы с дисбалансом классов используем технику взвешивания классов
from sklearn.utils import class_weight

# Вычисление весов классов
class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(Y),
    y=Y
)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

print(f"Веса классов: {class_weight_dict}")

Веса классов: {0: np.float64(0.5620503597122302), 1: np.float64(4.528985507246377)}


In [113]:
# Разделение на тренировочную и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

# Масштабирование признаков (особенно важно для SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Тренировочная выборка: {X_train_scaled.shape}")
print(f"Тестовая выборка: {X_test_scaled.shape}")

Тренировочная выборка: (1000, 538)
Тестовая выборка: (250, 538)


In [114]:
# Создание и обучение SVM модели
svm_model = SVC(
    kernel='rbf',  # Радиальная базисная функция
    C=1.0,         # Параметр регуляризации
    gamma='scale', # Коэффициент для RBF ядра
    class_weight=class_weight_dict,  # Балансировка классов
    probability=True,  # Для получения вероятностей
    random_state=42
)

# Обучение модели
svm_model.fit(X_train, y_train)

# Предсказания
y_pred_svm = svm_model.predict(X_test_scaled)
y_pred_proba_svm = svm_model.predict_proba(X_test_scaled)

# Оценка модели
print("=== SVM Модель ===")
print(f"Точность: {accuracy_score(y_test, y_pred_svm):.4f}")
print("\nМатрица ошибок:")
print(confusion_matrix(y_test, y_pred_svm))
print("\nОтчет по классификации:")
print(classification_report(y_test, y_pred_svm))

=== SVM Модель ===
Точность: 0.1120

Матрица ошибок:
[[  0 222]
 [  0  28]]

Отчет по классификации:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       222
           1       0.11      1.00      0.20        28

    accuracy                           0.11       250
   macro avg       0.06      0.50      0.10       250
weighted avg       0.01      0.11      0.02       250



e:\BioInformatics\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
e:\BioInformatics\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
e:\BioInformatics\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\BioInformatics\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.sh

In [115]:
# Создание и обучение Random Forest модели
rf_model = RandomForestClassifier(
    n_estimators=100,      # Количество деревьев
    max_depth=None,        # Максимальная глубина деревьев
    min_samples_split=2,   # Минимальное количество samples для разделения
    min_samples_leaf=1,    # Минимальное количество samples в листе
    class_weight=class_weight_dict,  # Балансировка классов
    random_state=42,
    n_jobs=-1  # Использование всех процессоров
)

# Обучение модели
rf_model.fit(X_train, y_train)  # Для RF масштабирование не обязательно

# Предсказания
y_pred_rf = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)

# Оценка модели
print("=== Random Forest Модель ===")
print(f"Точность: {accuracy_score(y_test, y_pred_rf):.4f}")
print("\nМатрица ошибок:")
print(confusion_matrix(y_test, y_pred_rf))
print("\nОтчет по классификации:")
print(classification_report(y_test, y_pred_rf))

=== Random Forest Модель ===
Точность: 0.9080

Матрица ошибок:
[[221   1]
 [ 22   6]]

Отчет по классификации:
              precision    recall  f1-score   support

           0       0.91      1.00      0.95       222
           1       0.86      0.21      0.34        28

    accuracy                           0.91       250
   macro avg       0.88      0.60      0.65       250
weighted avg       0.90      0.91      0.88       250



In [116]:
from sklearn.model_selection import GridSearchCV, cross_val_score

# Кросс-валидация для SVM
svm_scores = cross_val_score(svm_model, X_train_scaled, y_train, cv=5, scoring='accuracy')
print(f"SVM Кросс-валидация (средняя точность): {svm_scores.mean():.4f} (+/- {svm_scores.std() * 2:.4f})")

# Кросс-валидация для Random Forest
rf_scores = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='accuracy')
print(f"RF Кросс-валидация (средняя точность): {rf_scores.mean():.4f} (+/- {rf_scores.std() * 2:.4f})")

# Подбор гиперпараметров для Random Forest
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

grid_search_rf = GridSearchCV(
    RandomForestClassifier(class_weight=class_weight_dict, random_state=42),
    param_grid_rf,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search_rf.fit(X_train, y_train)
print(f"Лучшие параметры RF: {grid_search_rf.best_params_}")
print(f"Лучшая точность RF: {grid_search_rf.best_score_:.4f}")

SVM Кросс-валидация (средняя точность): 0.9280 (+/- 0.0196)
RF Кросс-валидация (средняя точность): 0.9020 (+/- 0.0080)
Лучшие параметры RF: {'max_depth': None, 'min_samples_split': 10, 'n_estimators': 50}
Лучшая точность RF: 0.9170


In [ ]:
import joblib

# Сохранение модели Random Forest
joblib.dump(rf_model, 'random_forest_model.pkl')

# Тестируем качество классификатора на наборе DeepSecE

In [ ]:
record_features("Test.fasta")

Признаки успешно записаны в Test.fasta.csv


In [107]:
test = read_data_csv('Datasets/Test.fasta.csv')
test = test.drop('id', axis=1)
print(test)
test_X = test.drop('label', axis=1)
test_Y = test['label']

         aac_A     aac_C     aac_D     aac_E     aac_F     aac_G     aac_H  \
0     0.076923  0.000000  0.059172  0.065089  0.065089  0.088757  0.023669   
1     0.089744  0.016484  0.056777  0.060440  0.051282  0.051282  0.032967   
2     0.086294  0.010152  0.055838  0.060914  0.030457  0.076142  0.000000   
3     0.090141  0.008451  0.081690  0.056338  0.033803  0.070423  0.019718   
4     0.082136  0.002053  0.059548  0.039014  0.034908  0.119097  0.016427   
...        ...       ...       ...       ...       ...       ...       ...   
1803  0.114583  0.011905  0.065476  0.053571  0.032738  0.056548  0.023810   
1804  0.082317  0.010671  0.064024  0.042683  0.030488  0.059451  0.045732   
1805  0.056630  0.008287  0.064917  0.087017  0.034530  0.062155  0.029006   
1806  0.105105  0.011011  0.064064  0.042042  0.031031  0.079079  0.020020   
1807  0.100548  0.016453  0.065814  0.058501  0.034735  0.089580  0.012797   

         aac_I     aac_K     aac_L  ...  qso_QSOgrant46  qso_QS

In [108]:
# Загрузка с помощью joblib
loaded_model = joblib.load('random_forest_model.pkl')

In [109]:
# Проверка работы загруженной модели
y_pred_loaded = loaded_model.predict(test_X)

# Оценка модели
print("=== Random Forest Модель ===")
print(f"Точность: {accuracy_score(test_Y, y_pred_loaded):.4f}")
print("\nМатрица ошибок:")
print(confusion_matrix(test_Y, y_pred_loaded))
print("\nОтчет по классификации:")
print(classification_report(test_Y, y_pred_loaded))

=== Random Forest Модель ===
Точность: 0.9270

Матрица ошибок:
[[1576    1]
 [ 131  100]]

Отчет по классификации:
              precision    recall  f1-score   support

           0       0.92      1.00      0.96      1577
           1       0.99      0.43      0.60       231

    accuracy                           0.93      1808
   macro avg       0.96      0.72      0.78      1808
weighted avg       0.93      0.93      0.91      1808

